# **Kaggle – DataTops®**
Tu TA ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Aspectos importantes
- Última submission:
    - Mañana: 17 de febrero a las 5pm
    - Tarde: 19 de febrero a las 5pm
- **Enlace de la competición**: https://www.kaggle.com/t/c5cc87b50c4b4770bdc8f5acbe15577d
- **Requisito**: Estar registrado en [Kaggle](https://www.kaggle.com/)

## Métrica:
El error cuadrático medio (RMSE, por sus siglas en inglés) es una medida de la desviación estándar de los residuos (errores de predicción). Los residuos representan la diferencia entre los valores observados y los valores predichos por el modelo. El RMSE indica qué tan dispersos están estos errores: cuanto menor es el RMSE, más cercanas están las predicciones a los valores reales. En otras palabras, el RMSE mide qué tan bien se ajusta la línea de regresión a los datos.


$$ RMSE = \sqrt{\frac{1}{n}\Sigma_{i=1}^{n}{\Big(\frac{d_i -f_i}{\sigma_i}\Big)^2}}$$


## 1. Librerías

In [1]:
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import urllib.request

## 2. Datos

In [2]:
# Para que funcione necesitas bajarte los archivos de datos de Kaggle
df_portatiles = pd.read_csv("./data/train.csv", index_col="laptop_ID")
df_portatiles.index.name=None

### 2.1 Exploración de los datos

In [3]:
df_portatiles.info() #Hay mucho string

<class 'pandas.core.frame.DataFrame'>
Index: 912 entries, 755 to 229
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           912 non-null    object 
 1   Product           912 non-null    object 
 2   TypeName          912 non-null    object 
 3   Inches            912 non-null    float64
 4   ScreenResolution  912 non-null    object 
 5   Cpu               912 non-null    object 
 6   Ram               912 non-null    object 
 7   Memory            912 non-null    object 
 8   Gpu               912 non-null    object 
 9   OpSys             912 non-null    object 
 10  Weight            912 non-null    object 
 11  Price_in_euros    912 non-null    float64
dtypes: float64(2), object(10)
memory usage: 92.6+ KB


In [64]:
df_portatiles.head(5) #Laptop ID --> índice de test csv

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


In [5]:
df_portatiles.tail()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
28,Dell,Inspiron 5570,Notebook,15.6,Full HD 1920x1080,Intel Core i5 8250U 1.6GHz,8GB,256GB SSD,AMD Radeon 530,Windows 10,2.2kg,800.00
1160,HP,Spectre Pro,2 in 1 Convertible,13.3,Full HD / Touchscreen 1920x1080,Intel Core i5 6300U 2.4GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.48kg,1629.00
78,Lenovo,IdeaPad 320-15IKBN,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,2TB HDD,Intel HD Graphics 620,No OS,2.2kg,519.00
23,HP,255 G6,Notebook,15.6,1366x768,AMD E-Series E2-9000e 1.5GHz,4GB,500GB HDD,AMD Radeon R2,No OS,1.86kg,258.00
229,Dell,Alienware 17,Gaming,17.3,IPS Panel Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,256GB SSD + 1TB HDD,Nvidia GeForce GTX 1060,Windows 10,4.42kg,2456.34


In [7]:
def describe_df(df):
    DATA_TYPE=df.dtypes
    MISSINGS=(df.isna().sum()/len(df)*100).sort_values(ascending=False)
    UNIQUE_VALUES=df.nunique()
    CARDIN=UNIQUE_VALUES/len(df)*100
    describe_df=pd.DataFrame([DATA_TYPE, MISSINGS, UNIQUE_VALUES, CARDIN])
    parametros=["DATA_TYPE", "MISSINGS (%)", "UNIQUE_VALUES", "CARDIN (%)"]
    describe_df.insert(0, "COL_N",parametros)
    return describe_df

In [8]:
describe_df(df_portatiles).T

,0,1,2,3
COL_N,DATA_TYPE,MISSINGS (%),UNIQUE_VALUES,CARDIN (%)
Company,object,0.0,19,2.083333
Product,object,0.0,480,52.631579
TypeName,object,0.0,6,0.657895
Inches,float64,0.0,17,1.864035
ScreenResolution,object,0.0,36,3.947368
Cpu,object,0.0,107,11.732456
Ram,object,0.0,9,0.986842
Memory,object,0.0,37,4.057018
Gpu,object,0.0,93,10.197368


In [9]:
for col in df_portatiles.columns:
    print(df_portatiles[col].value_counts(normalize=True)*100)
    print("-------------------------------------------------------")

Company
Lenovo       22.149123
Dell         21.600877
HP           21.271930
Asus         13.267544
Acer          8.114035
MSI           4.057018
Toshiba       3.728070
Apple         1.864035
Razer         0.657895
Mediacom      0.657895
Samsung       0.548246
Microsoft     0.548246
Xiaomi        0.328947
Huawei        0.219298
Chuwi         0.219298
Google        0.219298
Vero          0.219298
Fujitsu       0.219298
LG            0.109649
Name: proportion, dtype: float64
-------------------------------------------------------
Product
XPS 13                                   2.521930
Inspiron 3567                            2.412281
Legion Y520-15IKBN                       1.644737
Vostro 3568                              1.535088
ProBook 450                              1.425439
                                           ...   
Inspiron 7773                            0.109649
ENVY -                                   0.109649
Latitude E7270                           0.109649
Rog GL55

In [10]:
df_portatiles.describe()

,Inches,Price_in_euros
count,912.000000,912.000000
mean,14.981579,1111.724090
std,1.436719,687.959172
min,10.100000,174.000000
25%,14.000000,589.000000
50%,15.600000,978.000000
75%,15.600000,1483.942500
max,18.400000,6099.000000


In [11]:
#Hacemos una copia del dataset:
df=df_portatiles.copy()

### 2.3 Definir X e y

In [23]:
#Nota: considerar cambiar el orden 2.3 <--> 2.4
X = df.drop(['Price_in_euros'], axis=1)
y = df['Price_in_euros'].copy()
X.shape

(912, 11)

### 2.4 Dividir X_train, X_test, y_train, y_test

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42)

## 3. Procesado de datos

Nuestro target es la columna `Price_in_euros`

In [28]:
X_train.columns

Index(['Company', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram',
       'Memory', 'Gpu', 'OpSys', 'Weight'],
      dtype='object')

In [26]:
#Eliminamos la columna Product ya que tiene demasiada cardinalidad
X_train.drop(columns=["Product"], inplace=True)
X_test.drop(columns=["Product"], inplace=True)

#Columna RAM: Quitamos "GB" y sustituimos por entero
X_train["Ram"]=X_train["Ram"].str.replace('GB', "").astype(int)
X_test["Ram"]=X_test["Ram"].str.replace("GB","")

#Columna weight: quitamos "kg" y lo convertimos a número decimal
X_train["Weight"] = X_train["Weight"].str.replace("kg", "").astype(float)
X_test["Weight"] = X_test["Weight"].str.replace("kg", "").astype(float)



In [34]:
X_train.isna().sum()

Company             0
TypeName            0
Inches              0
ScreenResolution    0
Cpu                 0
Ram                 0
Memory              0
Gpu                 0
OpSys               0
Weight              0
dtype: int64

In [44]:
len(X_train)+183

912

In [48]:
#Extraemos los números correspondientes al ancho y alto de la resolución:
resolucion_train = X_train["ScreenResolution"].str.extract(r"(\d+)x(\d+)")
resolucion_test = X_test["ScreenResolution"].str.extract(r"(\d+)x(\d+)")

X_train["X_res"] = resolucion_train[0].astype(int)
X_train["Y_res"] = resolucion_train[1].astype(int)

X_test["X_res"] = resolucion_test[0].astype(int)
X_test["Y_res"] = resolucion_test[1].astype(int)

#Creamos la columna de pixeles 
X_train["Pixels"]=X_train["X_res"] * X_train["Y_res"]
X_test["Pixels"]=X_test["X_res"] * X_test["Y_res"]

#Eliminamos X_res e Y_res:
X_train.drop(columns=["X_res", "Y_res"], inplace=True)
X_test.drop(columns=["X_res", "Y_res"], inplace=True)


In [55]:
#Creamos variables binarias para las características extra
X_train['IPS'] = X_train['ScreenResolution'].apply(lambda x: 1 if 'IPS' in x else 0)
X_test['IPS'] = X_test['ScreenResolution'].apply(lambda x: 1 if 'IPS' in x else 0)
X_train['Touchscreen'] = X_train['ScreenResolution'].apply(lambda x: 1 if 'Touchscreen' in x else 0)
X_test['Touchscreen'] = X_test['ScreenResolution'].apply(lambda x: 1 if 'Touchscreen' in x else 0)

In [58]:
#Variable CPU: Extraemos el número que está por delante de "GHz":
X_train["Cpu_GHz"] = X_train["Cpu"].str.extract(r'(\d+\.?\d*)GHz').astype(float)
X_test["Cpu_GHz"] = X_test["Cpu"].str.extract(r'(\d+\.?\d*)GHz').astype(float)

In [61]:
def procesar_cpu(texto):
    if "Intel Core i7" in texto:
        return "Intel Core i7"
    elif "Intel Core i5" in texto:
        return "Intel Core i5"
    elif 'Intel Core i3' in texto:
        return "Intel Core i3"
    elif "Intel" in texto:
        return "Other Intel Processor" # Celeron, Pentium, etc.
    else:
        return "AMD Processor"

X_train["Cpu_Brand"] = X_train["Cpu"].apply(procesar_cpu)
X_test["Cpu_Brand"] = X_test["Cpu"].apply(procesar_cpu)

In [62]:
X_train.Cpu_Brand.value_counts(normalize=True)

Cpu_Brand
Intel Core i7            0.410151
Intel Core i5            0.318244
Other Intel Processor    0.127572
Intel Core i3            0.101509
AMD Processor            0.042524
Name: proportion, dtype: float64

In [ ]:
#Variable Memory:
# Aplicamos los cambios a ambos sets
for df_split in [X_train, X_test]:
    # 1. Limpieza básica y estandarización
    df_split["Memory"] = df_split["Memory"].astype(str).replace("\.0", "", regex=True)
    df_split["Memory"] = df_split["Memory"].str.replace("GB", "")
    df_split["Memory"] = df_split["Memory"].str.replace("TB", "000") # Convertimos TB a GB aproximados

    # 2. Creamos columnas binarias para los tipos más comunes
    df_split["Layer1HDD"] = df_split["Memory"].apply(lambda x: 1 if "HDD" in x else 0)
    df_split["Layer1SSD"] = df_split["Memory"].apply(lambda x: 1 if "SSD" in x else 0)
    
    # 3. Extraemos la cantidad numérica
    # Usamos regex para sacar solo los números antes de cualquier texto
    df_split["Memory_Amount"] = df_split["Memory"].str.extract("(\d+)").astype(float)

# Verificamos el resultado en X_train
print(X_train[["Memory", "Memory_Amount", "Layer1SSD", "Layer1HDD"]].head()

In [66]:
X_train.columns

Index(['Company', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram',
       'Memory', 'Gpu', 'OpSys', 'Weight', 'Pixels', 'IPS', 'Touchscreen',
       'Cpu_GHz', 'Cpu_Brand'],
      dtype='object')

In [27]:
X_train.describe()

,Inches,Ram,Weight
count,729.000000,729.000000,729.000000
mean,15.002881,8.233196,2.033752
std,1.428711,4.986294,0.668108
min,10.100000,2.000000,0.690000
25%,14.000000,4.000000,1.500000
50%,15.600000,8.000000,2.040000
75%,15.600000,8.000000,2.300000
max,17.300000,64.000000,4.700000


In [ ]:
#Matriz de correlación. Solo numéricas
X_train.corr(numeric_only=True)

In [ ]:
X_train["Ram"]=X_train["Ram"].astype(int)

In [ ]:
X_train["Ram"].value_counts(dropna=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,8))
sns.heatmap(
    X_train.corr(numeric_only=True),
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)
plt.title("Matriz de correlaciones (dataset limpio)")
plt.show()

-----------------------------------------------------------------------------------------------------------------

## 4. Modelado

### 4.1 Baseline de modelos


In [ ]:
features=["Inches", "Ram"]
rf=RandomForestRegressor(random_state=42)

rf.fit(X_train[features],y_train)



In [ ]:
y_pred=rf.predict(X_train[features])

### 4.2 Sacar métricas, valorar los modelos

Recuerda que en la competición se va a evaluar con la métrica de ``RMSE``.

In [ ]:
from sklearn.metrics import mean_squared_error

RMSE=np.sqrt(mean_squared_error(y_train,y_pred))
RMSE

### 4.3 Optimización (up to you 🫰🏻)

-----------------------------------------------------------------

## Una vez listo el modelo, toca predecir ``test.csv``

**RECUERDA: APLICAR LAS TRANSFORMACIONES QUE HAYAS REALIZADO EN `train.csv` a `test.csv`.**


Véase:
- Estandarización/Normalización
- Eliminación de Outliers
- Eliminación de columnas
- Creación de columnas nuevas
- Gestión de valores nulos
- Y un largo etcétera de técnicas que como Data Scientist hayas considerado las mejores para tu dataset.

## 1. Carga los datos de `test.csv` para predecir.


In [ ]:
#RECORDAR HACER LAS MISMAS TRANSFORMACIONES

In [ ]:
X_pred = pd.read_csv("./data/test.csv",index_col="laptop_ID")
df.index.name=None #establecer mejor laptop_id como indice
X_pred.head()

In [ ]:
X_pred.tail()

In [ ]:
X_pred.info()

 ## 2. Replicar el procesado para ``test.csv``

In [ ]:
X_pred

In [ ]:
X_pred["Ram"]=X_pred["Ram"].str.replace("GB","")
X_pred["Ram"]=X_pred["Ram"].astype(int)

In [ ]:
#a=X_pred["laptop_id"]

In [ ]:
model=rf

predictions_submit = model.predict(X_pred[features])
predictions_submit

**¡OJO! ¿Por qué me da error?**

IMPORTANTE:

- SI EL ARRAY CON EL QUE HICISTEIS `.fit()` ERA DE 4 COLUMNAS, PARA `.predict()` DEBEN SER LAS MISMAS
- SI AL ARRAY CON EL QUE HICISTEIS `.fit()` LO NORMALIZASTEIS, PARA `.predict()` DEBÉIS NORMALIZARLO
- TODO IGUAL SALVO **BORRAR FILAS**, EL NÚMERO DE ROWS SE DEBE MANTENER EN ESTE SET, PUES LA PREDICCIÓN DEBE TENER **391 FILAS**, SI O SI

**Entonces, si al cargar los datos de ``train.csv`` usaste `index_col=0`, ¿tendré que hacer lo también para el `test.csv`?**

In [ ]:
# ¿Qué opináis? Si hay que hacerlo
# ¿Sí, no?

![wow.jpeg](attachment:wow.jpeg)

## 3. **¿Qué es lo que subirás a Kaggle?**

**Para subir a Kaggle la predicción esta tendrá que tener una forma específica.**

En este caso, la **MISMA** forma que `sample_submission.csv`.

In [ ]:
sample = pd.read_csv("data/sample_submission.csv") #ïndice laptop ID

In [ ]:
sample #Formato que hay que subir

In [ ]:
sample.head()

In [ ]:
sample.shape

## 4. Mete tus predicciones en un dataframe llamado ``submission``.

In [ ]:
#¿Cómo creamos la submission?
submission = pd.DataFrame({"laptop_ID": X_pred.index , "Price_in_euros" : predictions_submit})

In [ ]:
submission.head()

In [ ]:
submission.shape

## 5. Pásale el CHEQUEADOR para comprobar que efectivamente está listo para subir a Kaggle.

In [ ]:
def chequeador(df_to_submit):
    """
    Esta función se asegura de que tu submission tenga la forma requerida por Kaggle.

    Si es así, se guardará el dataframe en un `csv` y estará listo para subir a Kaggle.

    Si no, LEE EL MENSAJE Y HAZLE CASO.

    Si aún no:
    - apaga tu ordenador,
    - date una vuelta,
    - enciendelo otra vez,
    - abre este notebook y
    - leelo todo de nuevo.
    Todos nos merecemos una segunda oportunidad. También tú.
    """
    if df_to_submit.shape == sample.shape:
        if df_to_submit.columns.all() == sample.columns.all():
            if df_to_submit.laptop_ID.all() == sample.laptop_ID.all():
                print("You're ready to submit!")
                df_to_submit.to_csv("submission.csv", index = False) #muy importante el index = False
                urllib.request.urlretrieve("https://www.mihaileric.com/static/evaluation-meme-e0a350f278a36346e6d46b139b1d0da0-ed51e.jpg", "gfg.png")
                img = Image.open("gfg.png")
                img.show()
            else:
                print("Check the ids and try again")
        else:
            print("Check the names of the columns and try again")
    else:
        print("Check the number of rows and/or columns and try again")
        print("\nMensaje secreto del TA: No me puedo creer que después de todo este notebook hayas hecho algún cambio en las filas de `test.csv`. Lloro.")

In [ ]:
chequeador(submission)